<a href="https://colab.research.google.com/github/AishwaryaChennadi/DataScience/blob/main/API_Testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import json
from datetime import datetime

LOG_FILE = "api_test_results.log"

# ---------------- API CALL ----------------

def get_data(url):
    try:
        response = requests.get(url, timeout=10)
        return response
    except Exception as e:
        return None, str(e)

# ---------------- VALIDATIONS ----------------

def validate_status(response):
    if response.status_code == 200:
        return True, "Status code is 200"
    return False, f"Expected 200, got {response.status_code}"


def validate_schema(response):
    try:
        data = response.json()
    except Exception:
        return False, "Response is not valid JSON"

    if not isinstance(data, list):
        return False, "Response is not a list"

    if len(data) == 0:
        return False, "Response list is empty"

    required_keys = ["userId", "id", "title", "body"]
    first_item = data[0]

    for key in required_keys:
        if key not in first_item:
            return False, f"Missing key: {key}"

    if not isinstance(first_item["userId"], int):
        return False, "userId is not int"
    if not isinstance(first_item["id"], int):
        return False, "id is not int"
    if not isinstance(first_item["title"], str):
        return False, "title is not string"
    if not isinstance(first_item["body"], str):
        return False, "body is not string"

    return True, "Schema validation passed"

# ---------------- LOGGING ----------------

def log_result(test_name, status, message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_line = f"{timestamp} | {test_name} | {status} | {message}\n"

    with open(LOG_FILE, "a") as file:
        file.write(log_line)

# ---------------- TEST RUNNER ----------------

def run_tests():
    url = "https://jsonplaceholder.typicode.com/posts"
    wrong_url = "https://jsonplaceholder.typicode.com/invalidendpoint"

    # ---- Valid Endpoint Tests ----
    response = get_data(url)

    if isinstance(response, tuple):
        log_result("API Call Test", "FAIL", response[1])
        return

    # Status Code Test
    status_result, message = validate_status(response)
    log_result("Status Code Test", "PASS" if status_result else "FAIL", message)

    # Schema Test
    schema_result, message = validate_schema(response)
    log_result("Schema Validation Test", "PASS" if schema_result else "FAIL", message)

    # ---- Invalid Endpoint Test ----
    bad_response = requests.get(wrong_url)

    if bad_response.status_code != 200:
        log_result(
            "Invalid Endpoint Test",
            "PASS",
            f"Handled correctly with status {bad_response.status_code}"
        )
    else:
        log_result(
            "Invalid Endpoint Test",
            "FAIL",
            "Expected non-200 status code"
        )

# ---------------- MAIN ----------------

if __name__ == "__main__":
    run_tests()

In [2]:
!cat api_test_results.log

2026-05-05 09:09:49 | Status Code Test | PASS | Status code is 200
2026-05-05 09:09:49 | Schema Validation Test | PASS | Schema validation passed
2026-05-05 09:09:49 | Invalid Endpoint Test | PASS | Handled correctly with status 404


In [ ]:
!cat api_test_results.log

In [3]:
!mkdir -p week2_api_testing/tests

In [9]:
%cd week2_api_testing

[Errno 2] No such file or directory: 'week2_api_testing'
/content/week2_api_testing


In [10]:
!pip install pytest requests

In [15]:
!pytest -v

============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/week2_api_testing
plugins: anyio-4.13.0, typeguard-4.5.1, langsmith-0.7.34
collected 3 items                                                              

tests/test_api.py::test_status_code PASSED                               [ 33%]
tests/test_api.py::test_schema_validation PASSED                         [ 66%]
tests/test_api.py::test_invalid_endpoint PASSED                          [100%]

============================== 3 passed in 0.10s ===============================


In [16]:
!cat api_test_results.log

2026-05-05 09:26:49 | INFO | Status Code Test PASSED
2026-05-05 09:26:49 | INFO | Schema Validation Test PASSED
2026-05-05 09:26:49 | INFO | Invalid Endpoint Test PASSED
